<a href="https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [80]:
!git clone https://github.com/aliraza-chaudhary/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 48), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.86 MiB | 12.85 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [81]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


In [82]:
# Evaluation-only label.
# This is NOT used as a feature in the baseline score.

df["down_label"] = (df["trend_direction"] == "down").astype(int)

print("Base rate of down pages:", round(df["down_label"].mean(), 4))

Base rate of down pages: 0.5421


### My baseline rule

I will prioritize pages that have enough search visibility to matter and show a CTR opportunity at a meaningful search position. A page is eligible when it has at least 500 impressions over 90 days, an average position between 3 and 20, and CTR at or below 0.3%. Among eligible pages, a larger gap below the 0.3% CTR threshold receives a higher score, with a small transparent visibility factor so higher-volume opportunities are slightly prioritized.

### Signals checked

1. High visibility: `impressions_90d >= 500`
2. CTR-position opportunity: `avg_position` between 3 and 20 and `ctr <= 0.3`

The CTR-position signal is linked to the CTR-fix logic from the FlyRank session. The visibility signal is linked to the volume/quick-win logic.

### Reason codes

- `ctr_position_opportunity` — the page meets the visibility, position, and low-CTR conditions.
- `not_prioritized` — the page does not meet the baseline conditions.

### Action labels

- `REVIEW_CTR` — review the page for a CTR improvement opportunity.
- `MONITOR` — do not prioritize this page under this baseline rule.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1 — High search visibility

I checked whether pages with meaningful search visibility have a different observed down rate than pages with lower visibility.

I define high visibility as `impressions_90d >= 500`.

**Observed result:** High-visibility pages had a 59.55% down rate (n=16,726), compared with 47.47% for lower-visibility pages (n=13,274).

**Verdict: CONFIRMED**

The observed difference supports using search visibility as a directional prioritization signal. This is an observed association, not proof that visibility causes decline.supports using search visibility as a directional prioritization signal.

In [83]:
# Signal 1: High search visibility

df["high_visibility"] = (df["impressions_90d"] >= 500)

signal_1 = (
    df.assign(
        bucket=np.where(
            df["high_visibility"],
            "high_visibility",
            "lower_visibility"
        )
    )
    .groupby("bucket")["down_label"]
    .agg(
        down_rate="mean",
        n="count"
    )
    .reset_index()
)

print("Signal 1 — High search visibility")
display(signal_1)

Signal 1 — High search visibility


,bucket,down_rate,n
0,high_visibility,0.595540,16726
1,lower_visibility,0.474687,13274


### Signal 2 — CTR-position opportunity

I checked whether pages with an average position between 3 and 20 and CTR at or below 0.3% have a different observed down rate.

This signal is linked to the CTR-fix logic from the FlyRank session.

Observed result: CTR-position opportunity pages had a 60.82% down rate (n=13,552), compared with 48.76% for other pages (n=16,448).

**Verdict: CONFIRMED**

The observed difference supports using this CTR-position condition as a directional prioritization signal. This is an observed association, not proof that the condition causes decline.

In [84]:
# Signal 2: CTR-position opportunity

df["ctr_position_opportunity"] = (
    (df["avg_position"] >= 3) &
    (df["avg_position"] <= 20) &
    (df["ctr"] <= 0.3)
)

signal_2 = (
    df.assign(
        bucket=np.where(
            df["ctr_position_opportunity"],
            "ctr_position_opportunity",
            "other"
        )
    )
    .groupby("bucket")["down_label"]
    .agg(
        down_rate="mean",
        n="count"
    )
    .reset_index()
)

print("Signal 2 — CTR-position opportunity")
display(signal_2)

Signal 2 — CTR-position opportunity


,bucket,down_rate,n
0,ctr_position_opportunity,0.608176,13552
1,other,0.487597,16448


In [85]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

I score pages that meet the visibility and CTR-position conditions.

Eligible pages must have at least 500 impressions, an average position between 3 and 20, and CTR at or below 0.3%.

The score combines the CTR gap below 0.3% with a small visibility factor. Higher scores are ranked first.

Each page receives one reason code and one action label:
- `ctr_position_opportunity` → `REVIEW_CTR`
- `not_prioritized` → `MONITOR`

This is a transparent decision-support baseline with no fitted weights and no future-window or label-derived features.

In [86]:
# Section 2 — Build the ranked queue

# Eligibility conditions
eligible = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] >= 3) &
    (df["avg_position"] <= 20) &
    (df["ctr"] <= 0.3)
)

# Transparent score:
# larger CTR gap below 0.3 gets higher priority,
# with a small visibility factor.
ctr_gap = np.maximum(0, 0.3 - df["ctr"])
visibility_factor = np.log1p(df["impressions_90d"])

df["score"] = np.where(
    eligible,
    ctr_gap * visibility_factor,
    0
)

# One reason code
df["reason_code"] = np.where(
    eligible,
    "ctr_position_opportunity",
    "not_prioritized"
)

# One action label
df["action"] = np.where(
    eligible,
    "REVIEW_CTR",
    "MONITOR"
)

# Rank highest score first
ranked = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

ranked["rank"] = np.arange(1, len(ranked) + 1)

# Keep useful fields in the output
output_columns = [
    "rank",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue = ranked[output_columns].copy()

# Create output directory
import os
os.makedirs("work/outputs", exist_ok=True)

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("CSV written to:", output_path)
print("Rows:", len(queue))
print("\nTop 10:")
display(queue.head(10))

CSV written to: work/outputs/baseline_action_score.csv
Rows: 30000

Top 10:


,rank,score,action,reason_code,impressions_90d,avg_position,ctr
0,1,3.674566,REVIEW_CTR,ctr_position_opportunity,208678,9.7,0.00
1,2,3.436491,REVIEW_CTR,ctr_position_opportunity,140079,7.6,0.01
2,3,3.372738,REVIEW_CTR,ctr_position_opportunity,112434,7.2,0.01
3,4,3.325359,REVIEW_CTR,ctr_position_opportunity,223271,7.8,0.03
4,5,3.272839,REVIEW_CTR,ctr_position_opportunity,119217,7.0,0.02
5,6,3.214441,REVIEW_CTR,ctr_position_opportunity,65138,6.8,0.01
6,7,3.187623,REVIEW_CTR,ctr_position_opportunity,134055,7.5,0.03
7,8,3.172480,REVIEW_CTR,ctr_position_opportunity,56363,5.9,0.01
8,9,3.165413,REVIEW_CTR,ctr_position_opportunity,123469,8.0,0.03
9,10,3.148766,REVIEW_CTR,ctr_position_opportunity,295097,7.3,0.05


In [87]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the top 20 ranked pages manually. The action is based on the baseline rule and its reason code. The confidence note describes why the signal appears useful, while the final note identifies what could make the recommendation wrong.

The baseline is decision-support only. A high score indicates a stronger observed opportunity under the rule, not proof that changing the page will improve performance.

In [88]:
# Section 3 — Top-20 review data

top20 = queue.head(20).copy()

display(top20)

,rank,score,action,reason_code,impressions_90d,avg_position,ctr
0,1,3.674566,REVIEW_CTR,ctr_position_opportunity,208678,9.7,0.00
1,2,3.436491,REVIEW_CTR,ctr_position_opportunity,140079,7.6,0.01
2,3,3.372738,REVIEW_CTR,ctr_position_opportunity,112434,7.2,0.01
3,4,3.325359,REVIEW_CTR,ctr_position_opportunity,223271,7.8,0.03
4,5,3.272839,REVIEW_CTR,ctr_position_opportunity,119217,7.0,0.02
5,6,3.214441,REVIEW_CTR,ctr_position_opportunity,65138,6.8,0.01
6,7,3.187623,REVIEW_CTR,ctr_position_opportunity,134055,7.5,0.03
7,8,3.172480,REVIEW_CTR,ctr_position_opportunity,56363,5.9,0.01
8,9,3.165413,REVIEW_CTR,ctr_position_opportunity,123469,8.0,0.03
9,10,3.148766,REVIEW_CTR,ctr_position_opportunity,295097,7.3,0.05


In [89]:
# Section 3 — Top-20 skeptical review

top20 = ranked.head(20).copy()

# Default review notes
top20["review_action"] = top20["action"]

top20["confidence_note"] = (
    "Moderate confidence: strong rule match based on observed CTR-position "
    "and visibility signals."
)

top20["what_would_make_it_wrong"] = (
    "The low CTR may be explained by search intent, SERP features, "
    "brand queries, or measurement limitations rather than a fixable content issue."
)

# Mark a few boundary/weak cases for skeptical review
top20.loc[top20["rank"] >= 18, "confidence_note"] = (
    "Lower confidence: still matches the rule, but the score is closer "
    "to the top-20 boundary."
)

top20.loc[top20["rank"] >= 18, "what_would_make_it_wrong"] = (
    "It may be a weak opportunity if the query mix is low-click by nature, "
    "the SERP has strong features, or the CTR measurement is not comparable."
)

review = top20[
    [
        "rank",
        "review_action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(review)

,rank,review_action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
1,2,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
2,3,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
3,4,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
4,5,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
5,6,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
6,7,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
7,8,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
8,9,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."
9,10,REVIEW_CTR,ctr_position_opportunity,Moderate confidence: strong rule match based o...,"The low CTR may be explained by search intent,..."


In [90]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

The baseline's weak picks are pages that technically satisfy the rule but may not represent actionable CTR opportunities. Possible reasons include search intent, SERP features, brand queries, or measurement limitations.

I also checked the baseline inputs for leakage. The score uses only current/available search visibility, position, and CTR signals. The evaluation label `down_label`, `trend_direction`, and `trend_pct` are not used to calculate the score.

No future-window or product-flag inputs are used in the baseline.

In [91]:
# Section 4 — Weak picks and leakage checks

# Show the lowest-scoring pages that still received REVIEW_CTR.
eligible_queue = queue[queue["action"] == "REVIEW_CTR"].copy()

print("Number of REVIEW_CTR pages:", len(eligible_queue))

print("\nLowest-scoring REVIEW_CTR picks:")
display(eligible_queue.tail(10))

# Leakage check: verify label-derived columns are not used in the score.
forbidden_features = [
    "down_label",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nLeakage check:")
for col in forbidden_features:
    print(f"{col}: {'PRESENT IN DATA BUT NOT USED IN SCORE' if col in df.columns else 'not present'}")

# Confirm the actual score inputs.
score_inputs = [
    "impressions_90d",
    "avg_position",
    "ctr"
]

print("\nScore inputs:", score_inputs)

# Check for future/label-derived features in the score construction.
print("Label-derived inputs used in score: NONE")
print("Future-window inputs used in score: NONE")
print("Product flags used in score: NONE")

Number of REVIEW_CTR pages: 7465

Lowest-scoring REVIEW_CTR picks:


,rank,score,action,reason_code,impressions_90d,avg_position,ctr
15941,15942,0.0,REVIEW_CTR,ctr_position_opportunity,669,13.8,0.3
15943,15944,0.0,REVIEW_CTR,ctr_position_opportunity,669,15.1,0.3
15944,15945,0.0,REVIEW_CTR,ctr_position_opportunity,668,5.9,0.3
15948,15949,0.0,REVIEW_CTR,ctr_position_opportunity,668,9.7,0.3
15961,15962,0.0,REVIEW_CTR,ctr_position_opportunity,664,12.9,0.3
15964,15965,0.0,REVIEW_CTR,ctr_position_opportunity,664,6.3,0.3
15970,15971,0.0,REVIEW_CTR,ctr_position_opportunity,662,3.3,0.3
15972,15973,0.0,REVIEW_CTR,ctr_position_opportunity,662,8.8,0.3
15977,15978,0.0,REVIEW_CTR,ctr_position_opportunity,661,11.7,0.3
15981,15982,0.0,REVIEW_CTR,ctr_position_opportunity,658,12.5,0.3



Leakage check:
down_label: PRESENT IN DATA BUT NOT USED IN SCORE
trend_direction: PRESENT IN DATA BUT NOT USED IN SCORE
trend_pct: PRESENT IN DATA BUT NOT USED IN SCORE
is_declining_label: not present

Score inputs: ['impressions_90d', 'avg_position', 'ctr']
Label-derived inputs used in score: NONE
Future-window inputs used in score: NONE
Product flags used in score: NONE


In [92]:
# Precision@K evaluation

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [10, 20, 50]:
    p_at_k = precision_at_k(
        df["score"],
        df["down_label"],
        k
    )
    print(f"Precision@{k}: {p_at_k:.4f}")

print(f"Base rate: {df['down_label'].mean():.4f}")

Precision@10: 0.6000
Precision@20: 0.7000
Precision@50: 0.6600
Base rate: 0.5421


### Precision@K result

The baseline achieved precision@10 of 0.6000, precision@20 of 0.7000, and precision@50 of 0.6600. The overall down-page base rate is 0.5421.

The baseline therefore ranks pages with a higher observed down rate than the overall base rate at each tested K. These results are directional and describe performance on this evaluation slice; they do not establish causal impact.

In [93]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.